In [1]:
import torch
import pandas as pd
import sys
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
sys.path.append('../../../../src/fluprofiler/')
from data.loaders import load_embedding
sys.path.append('../../')
sys.path.append('../../../../src/fluprofiler/models/')
from experiment_tools import fluProfiler_Dataset, generate_matrix
# from architectures import fluProfiler_Model

In [2]:
# 完全复制训练脚本的数据加载逻辑
data_path = '/home/chenyh/workspace/fluProfiler/data/reverse_test/'
season_path = 'processed/test_2024SH/'

test_data = pd.read_csv(data_path + season_path + 'test.csv')

# 创建数据集（与训练脚本完全一致）
test_dataset = fluProfiler_Dataset(test_data, add_special_token=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# 使用所有数据的合并来确保加载所有需要的embeddings（与训练脚本一致）
embedding_df = test_data


In [3]:
device = torch.device('cuda:4')
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
# 与训练脚本一致：不使用map_location，让embeddings保持在CPU上，使用时再to(device)
emb_dict = load_embedding(data_path + 'embedding', files=sequence_names)

Loading tensor: 100%|██████████| 1162/1162 [00:43<00:00, 26.66file/s]


In [4]:
model = torch.load('/home/chenyh/workspace/fluProfiler/runs/reverse_tests/2024SH/value_attention/20260205_212614__v0_1__pid989778/checkpoints/2026-02-05_22-34-56.pth',
weights_only=False,map_location=device)

In [5]:
import torch.nn.functional as F

prediction_ls_valid = []
reference_ls_valid = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in test_loader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_valid.extend(output.view(-1).tolist())
    reference_ls_valid.extend(labels.tolist())


In [ ]:
sys.path.append('../../../../src/fluprofiler/')
from evaluation.metrics import print_exams
print_exams(reference_ls_valid, prediction_ls_valid)

MAE: 0.96471
MSE: 1.69240
pearson correlation: 0.38010
spearman correlation: 0.37953
R2_score: 0.12861


(0.9647090640208946,
 1.692399374582753,
 PearsonRResult(statistic=0.3800993682823907, pvalue=5.554237490518463e-263),
 SignificanceResult(statistic=0.37952580255259755, pvalue=3.9418351561001257e-262),
 0.12861294407374713)

: 